<a href="https://colab.research.google.com/github/sznajder/Notebooks/blob/master/PQuantML_tutorial_Keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setting up training environment

In [ ]:
!pip install pquant-ml[tensorflow]
!pip install git+https://github.com/enlupi/hls4ml.git@pquant_PQLayers

In [ ]:
import os
import random

os.environ['KERAS_BACKEND'] = 'tensorflow'

import keras
import numpy as np
import tensorflow as tf
from matplotlib import pyplot as plt
from pquant.layers import PQDense
from pquant.activations import PQActivation
from pquant import dst_config
from pquant import get_ebops
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
set_seed(42)

## Fetch the Jet tagging dataset from Open ML

In [ ]:
import pickle as pkl
from pathlib import Path
import h5py as h5
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def get_data(data_path: Path | str, seed=42, src='openml'):
    data_path = Path(data_path)
    assert src in ('openml', 'cernbox')
    if src == 'openml':
        if not os.path.exists(data_path):
            print('Downloading data...')
            data = fetch_openml('hls4ml_lhc_jets_hlf')
            X, y = np.array(data['data']), data['target']
            codecs = {'g': 0, 'q': 1, 'w': 2, 'z': 3, 't': 4}
            y = np.array([codecs[i] for i in y])
            os.makedirs(data_path.parent, exist_ok=True)
            with h5.File(data_path, 'w') as f:
                f.create_dataset('X', data=X, compression='gzip')
                f.create_dataset('y', data=y, compression='gzip')
        else:
            print("data", data_path)
            with h5.File(data_path, 'r') as f:
                X = np.array(f['X'])
                y = np.array(f['y'])
    else:
        with h5.File(data_path, 'r') as f:
            raw = np.array(f['t_allpar_new'])
        df = pd.DataFrame(raw)
        feature_names = [
            'j_zlogz',
            'j_c1_b0_mmdt',
            'j_c1_b1_mmdt',
            'j_c1_b2_mmdt',
            'j_c2_b1_mmdt',
            'j_c2_b2_mmdt',
            'j_d2_b1_mmdt',
            'j_d2_b2_mmdt',
            'j_d2_a1_b1_mmdt',
            'j_d2_a1_b2_mmdt',
            'j_m2_b1_mmdt',
            'j_m2_b2_mmdt',
            'j_n2_b1_mmdt',
            'j_n2_b2_mmdt',
            'j_mass_mmdt',
            'j_multiplicity',
        ]
        labels = ['j_g', 'j_q', 'j_w', 'j_z', 'j_t']
        X, y = df[feature_names].to_numpy(), df[labels].to_numpy().argmax(axis=1)

    X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

    X_train_val, X_test, y_train_val, y_test = X_train_val.astype(np.float32), X_test.astype(np.float32), y_train_val, y_test

    scaler = StandardScaler()
    X_train_val = scaler.fit_transform(X_train_val)
    X_test = scaler.transform(X_test)

    X_train_val = X_train_val.astype(np.float32)
    y_train_val = y_train_val.astype(np.int32)
    X_test = X_test.astype(np.float32)
    y_test = y_test.astype(np.int32)

    N_train = int(0.9 * len(X_train_val))
    X_train, X_val = X_train_val[:N_train], X_train_val[N_train:]
    y_train, y_val = y_train_val[:N_train], y_train_val[N_train:]

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)



(X_train, y_train), (X_val, y_val), (X_test, y_test) = get_data(Path('/tmp/inp_data.zst'))

## Create the dataloaders for the dataset

In [ ]:
from hgq.utils.sugar import Dataset

def create_dataloaders(batch_size=1024):
  _y_train = keras.utils.to_categorical(y_train, 5)
  _y_val = keras.utils.to_categorical(y_val, 5)
  _y_test = keras.utils.to_categorical(y_test, 5)
  dataset_train = Dataset(X_train, _y_train, batch_size=batch_size, device='gpu:0', workers=4)
  dataset_val = Dataset(X_val, _y_val, batch_size=batch_size, device='gpu:0', workers=4)
  dataset_test = Dataset(X_test, _y_test, batch_size=batch_size, device='gpu:0')
  return dataset_train, dataset_val, dataset_test

## Define the PQuantML config and construct the model
The architecture is a dense network with 3 hidden layers with 64,32,32 units, followed by an output layer of 5 units. In PyTorch, no PQActivation class is needed for ReLU.

In [ ]:
def build_model(config, in_out_quant_bits=(1.,4.,11.), prune_output=True):
        inp = keras.layers.Input(shape=((16,)))
        x = PQDense(config, 64, in_quant_bits=in_out_quant_bits)(inp)
        x = PQActivation(config, "relu", quantize_input=False)(x)
        x = PQDense(config, 32)(x)
        x = PQActivation(config, "relu", quantize_input=False)(x)
        x = PQDense(config, 32)(x)
        x = PQActivation(config, "relu", quantize_input=False)(x)
        x = keras.layers.ReLU()(x)
        out = PQDense(config, 5, out_quant_bits=in_out_quant_bits, quantize_output=True, enable_pruning=prune_output)(x)

        return keras.Model(inputs=inp, outputs=out)

# DST with fixed point quantization

## Create the config

In [ ]:
from pquant import dst_config
# Training data range between -6.18 and 8.96, set 4 integer bits for model input quantizer above to cover whole range
config = dst_config()
config.training_parameters.epochs = 100
# 8 bit data and weights
# For data default k=0, weights k=1
config.quantization_parameters.default_data_integer_bits = 3.
config.quantization_parameters.default_data_fractional_bits = 5.
config.quantization_parameters.default_weight_integer_bits = 0.
config.quantization_parameters.default_weight_fractional_bits = 7.
config.quantization_parameters.overflow_mode_data = "SAT"
config.quantization_parameters.overflow_mode_parameters = "SAT"
#config.pruning_parameters.enable_pruning = False
# DST hyperparameter, higher value means higher sparsity
config.pruning_parameters.alpha = 1e-4
config

## PQuantCallback

With Keras model the switching between different stages is handled by PQuantCallback during model.fit. The number of epochs per stage are read from the config, or they can be provided as parameters when creating the callback. The callback includes options to also gather statistics about EPOBs and remaining weights.

In [ ]:
# Create callbacks
from pquant.core.keras.train import PQuantCallback
from hgq.utils.sugar import PBar
pbar = PBar('loss: {loss:.3f}/{val_loss:.3f} - acc: {accuracy:.3f}/{val_accuracy:.3f} - Remaining weights: {remaining_weights} - Stage: {stage}')
pqcb = PQuantCallback(config, True, True)

## Train the model

In [ ]:
from pquant import train_model
BATCH_SIZE = 1024 # Higher for tutorial, should be lower e.g. 1024
dataset_train, dataset_val, dataset_test = create_dataloaders(BATCH_SIZE)
loss = keras.losses.CategoricalCrossentropy(from_logits=True)


model = build_model(config)
#  Better performance with DST methods. Exclude learned threshold tensor from weight decay
opt = keras.optimizers.Adam(learning_rate=1e-3, weight_decay=1e-4)
opt.exclude_from_weight_decay(var_names=["threshold"])
model.compile(opt, loss, metrics=['accuracy'], jit_compile=False)
callbacks = [pqcb, pbar]
epochs = config.training_parameters.pretraining_epochs + config.training_parameters.epochs + config.training_parameters.fine_tuning_epochs

from datetime import datetime

start = datetime.now()
history = model.fit(dataset_train, epochs=epochs, batch_size=BATCH_SIZE, validation_data=dataset_test, verbose=0, callbacks=callbacks)
end = datetime.now()
print("Finished training, took", end-start)

## Apply compression to weights one more time
The weights and biases during training are in their non-compressed state, only going through compression during the forward pass. Run the function below to compress the weights and biases one more time, and override the previous non-compressed state of weights and biases. This is done by default automatically at the end of PQuantCallBack

In [ ]:
from pquant import apply_final_compression
apply_final_compression(model)

## Convert the model to FPGA firmware with hls4ml


The hls4ml converter extracts the quantization parameters from the PQLayers during the conversion.

In [ ]:
from hls4ml.converters import convert_from_keras_model
from hls4ml.utils import config_from_keras_model
output_dir = f"example_result_dst"
os.makedirs(output_dir, exist_ok = True)
#model.save(f"{output_dir}/model.keras")
io_type = 'io_parallel'
backend = 'vitis'
target = 'xcvu13p-flga2577-2-e'
hls_config = config_from_keras_model(
                model,
                granularity='name',
                backend=backend,
            )
hls_model = convert_from_keras_model(
                model,
                io_type=io_type,
                output_dir=output_dir,
                backend=backend,
                hls_config=hls_config,
                part=target,
            )
hls_model.compile()
res_hls = hls_model.predict(np.ascontiguousarray(X_test))
test_accuracy_hls4ml_config = np.mean((np.argmax(res_hls, axis=1) == y_test))


p_keras = model.predict(dataset_test, verbose=0)
p_hls = hls_model.predict(np.ascontiguousarray(X_test))
test_accuracy_hls4ml = np.mean((np.argmax(p_hls, axis=1) == y_test))
test_accuracy_keras = np.mean((np.argmax(p_keras, axis=1) == y_test))

print(f"Accuracies: hls4ml={test_accuracy_hls4ml_config}, keras={test_accuracy_keras}")

## Make some plots from values gathered during training

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,3, figsize=(20,5))
ax[0].plot(history.history["val_accuracy"], label="Accuracy")
ax[0].set_ylim(0.65, 0.78)
ax[0].legend()

ax[1].plot(history.history["ebops"], label="EBOPs")
ax[1].legend()

ax[2].plot(history.history["remaining_weights"], label="Remaining weights %")
ax[2].legend()
plt.suptitle("DST")
plt.tight_layout()
plt.show()

# PDP (structured)

## Create the config

In [ ]:
from pquant import pdp_config
config = pdp_config()
config.training_parameters.epochs = 100
config.training_parameters.pretraining_epochs = 20
config.training_parameters.fine_tuning_epochs = 100
# 8 bit data and weights
# For data default k=0
config.quantization_parameters.default_data_integer_bits = 3.
config.quantization_parameters.default_data_fractional_bits = 5.
config.quantization_parameters.default_weight_integer_bits = 0.
config.quantization_parameters.default_weight_fractional_bits = 7.
config.quantization_parameters.overflow_mode_data = "SAT"
config.quantization_parameters.overflow_mode_parameters = "SAT"
config.quantization_parameters.granularity = "per_channel"
config.pruning_parameters.structured_pruning = True
config.pruning_parameters.sparsity = 0.8
# Reach target sparsity around epoch 67: 0.015 * 67 > 1
config.pruning_parameters.epsilon = 0.015

## Train the model

In [ ]:
# Create callbacks
from pquant.core.keras.train import PQuantCallback
from hgq.utils.sugar import PBar
pbar = PBar('loss: {loss:.3f}/{val_loss:.3f} - acc: {accuracy:.3f}/{val_accuracy:.3f} - Remaining weights: {remaining_weights} - Stage: {stage}')
pqcb = PQuantCallback(config, True, True)

In [ ]:
BATCH_SIZE = 1024 # Higher for tutorial, should be lower e.g. 1024
dataset_train, dataset_val, dataset_test = create_dataloaders(BATCH_SIZE)
loss = keras.losses.CategoricalCrossentropy(from_logits=True)


model = build_model(config, prune_output=False) # Structured pruning, don't prune output layer
#  Better performance with DST methods. Exclude learned threshold tensor from weight decay
opt = keras.optimizers.Adam(learning_rate=1e-3, weight_decay=1e-4)
opt.exclude_from_weight_decay(var_names=["threshold"])
model.compile(opt, loss, metrics=['accuracy'], jit_compile=False)
callbacks = [pqcb, pbar]
epochs = config.training_parameters.pretraining_epochs + config.training_parameters.epochs + config.training_parameters.fine_tuning_epochs

from datetime import datetime

start = datetime.now()
history = model.fit(dataset_train, epochs=epochs, batch_size=BATCH_SIZE, validation_data=dataset_test, verbose=0, callbacks=callbacks)
end = datetime.now()
print("Finished training, took", end-start)

## Apply compression to weights one more time
The weights and biases during training are in their non-compressed state, only going through compression during the forward pass. Run the function below to compress the weights and biases one more time, and override the previous non-compressed state of weights and biases. This is done by default automatically at the end of PQuantCallBack

In [ ]:
from pquant import apply_final_compression
apply_final_compression(model)

## Convert the model to FPGA firmware with hls4ml
The hls4ml converter extracts the quantization parameters from the PQLayers during the conversion.

In [ ]:
output_dir = f"example_result_dst"
os.makedirs(output_dir, exist_ok = True)
#model.save(f"{output_dir}/model.keras")
io_type = 'io_parallel'
backend = 'vitis'
target = 'xcvu13p-flga2577-2-e'
hls_config = config_from_keras_model(
                model,
                granularity='name',
                backend=backend,
            )
hls_model = convert_from_keras_model(
                model,
                io_type=io_type,
                output_dir=output_dir,
                backend=backend,
                hls_config=hls_config,
                part=target,
            )
hls_model.compile()
res_hls = hls_model.predict(np.ascontiguousarray(X_test))
test_accuracy_hls4ml_config = np.mean((np.argmax(res_hls, axis=1) == y_test))


p_keras = model.predict(dataset_test, verbose=0)
p_hls = hls_model.predict(np.ascontiguousarray(X_test))
test_accuracy_hls4ml = np.mean((np.argmax(p_hls, axis=1) == y_test))
test_accuracy_keras = np.mean((np.argmax(p_keras, axis=1) == y_test))

print(f"Accuracies: hls4ml={test_accuracy_hls4ml_config}, keras={test_accuracy_keras}")

## Make some plots from values gathered during training

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,3, figsize=(20,5))
ax[0].plot(history.history["val_accuracy"], label="Accuracy")
ax[0].set_ylim(0.65, 0.78)
ax[0].legend()

ax[1].plot(history.history["ebops"], label="EBOPs")
ax[1].legend()

ax[2].plot(history.history["remaining_weights"], label="Remaining weights %")
ax[2].legend()
plt.suptitle("PDP")
plt.tight_layout()
plt.show()